# Capa de modelado - Regresion y clasificacion

Esta capa usa las decisiones de EDA/preprocesamiento para entrenar modelos sin fuga de datos. Se trabaja con dos tareas:

- Clasificacion: usar `koi_disposition` para predecir si una senal queda como `CONFIRMED` o `NO_CONFIRMED`.
- Regresion: predecir `log1p(koi_prad)` como transformacion del radio planetario.

La clasificacion compara algoritmos vistos en clase: regresion logistica, k-NN, arbol de decision y Naive Bayes. La evaluacion usa las metricas de la diapositiva: Accuracy, Precision, Recall, F1 y matriz de confusion. En regresion se comparan modelos lineales: regresion lineal, Ridge y Lasso.


In [ ]:
from __future__ import annotations

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

PLOTLY_TEMPLATE = "plotly_white"
pd.set_option("display.max_columns", 120)


## 1. Cargar datos desde el warehouse

La capa de modelado consume el DuckDB creado por la capa de datos. Si `data/warehouse/exoplanets.duckdb` no existe, primero se debe ejecutar `02_capa_datos_warehouse.ipynb`.


In [ ]:
CANDIDATE_DATA_DIRS = [Path("data"), Path("mineria") / "data"]
DATA_DIR = next(
    (
        data_dir
        for data_dir in CANDIDATE_DATA_DIRS
        if (data_dir / "warehouse" / "exoplanets.duckdb").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontro data/warehouse/exoplanets.duckdb. Ejecuta primero 02_capa_datos_warehouse.ipynb."
    )

DB_PATH = DATA_DIR / "warehouse" / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

kepler = con.execute("""
SELECT
    star_id AS kepid,
    candidate_id AS kepoi_name,
    kepler_name,
    koi_disposition,
    planet_radius_earth AS koi_prad,
    orbital_period_days AS koi_period,
    transit_duration_hours AS koi_duration,
    transit_depth_ppm AS koi_depth,
    equilibrium_temp_k AS koi_teq,
    insolation_flux AS koi_insol,
    model_snr AS koi_model_snr,
    impact_parameter AS koi_impact,
    koi_steff,
    koi_slogg,
    koi_srad,
    ra,
    dec,
    koi_kepmag,
    star_temp_band,
    planet_radius_band,
    ra_bin,
    dec_bin
FROM v_koi_observations
""").df()

print("Datos de modelado cargados desde warehouse:", DB_PATH)
kepler.shape


## 2. Variables y columnas excluidas

Se excluyen identificadores y columnas con fuga. Los identificadores no describen fisicamente al candidato, y las columnas de score/flags contienen informacion demasiado cercana al dictamen final.


In [ ]:
columnas_identificacion = ["kepid", "kepoi_name", "kepler_name"]
columnas_fuga = [
    "koi_score",
    "koi_pdisposition",
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
]

features_clasificacion = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

# Para regresion se excluyen koi_depth y koi_srad porque estan fisicamente muy cerca del calculo de koi_prad.
features_regresion = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "ra",
    "dec",
    "koi_kepmag",
]

target_clasificacion = "koi_disposition"
target_regresion = "koi_prad"

pd.DataFrame({
    "grupo": ["ids_no_predictoras", "fuga_no_usar", "features_clasificacion", "features_regresion"],
    "columnas": [
        ", ".join(columnas_identificacion),
        ", ".join(columnas_fuga),
        ", ".join(features_clasificacion),
        ", ".join(features_regresion),
    ],
})


## 3. Preprocesamiento sin data leakage

El imputador y el scaler van dentro del `Pipeline`. Por eso se ajustan con train y despues se aplican a test.


In [ ]:
def make_numeric_preprocessor(features: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "numericas",
                Pipeline(
                    steps=[
                        ("imputar_mediana", SimpleImputer(strategy="median")),
                        ("escalar", StandardScaler()),
                    ]
                ),
                features,
            )
        ],
        remainder="drop",
    )


## 4. Clasificacion: `koi_disposition`

La columna original es `koi_disposition`. Para usar exactamente las metricas binarias de la diapositiva, se convierte en:

- `CONFIRMED`: positivo.
- `NO_CONFIRMED`: agrupa `CANDIDATE` y `FALSE POSITIVE`.

Se usa `stratify=y` para conservar la proporcion de clases en train y test.


In [ ]:
clasificacion_df = kepler[features_clasificacion + [target_clasificacion]].dropna(subset=[target_clasificacion]).copy()
clasificacion_df["target_confirmed"] = np.where(
    clasificacion_df[target_clasificacion] == "CONFIRMED",
    "CONFIRMED",
    "NO_CONFIRMED",
)

X_clf = clasificacion_df[features_clasificacion]
y_clf = clasificacion_df["target_confirmed"]

print("Distribucion del target binario")
display(y_clf.value_counts().rename_axis("clase").reset_index(name="n"))

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf,
)

pd.DataFrame({
    "particion": ["X_train", "X_test", "y_train", "y_test"],
    "shape": [X_train_clf.shape, X_test_clf.shape, y_train_clf.shape, y_test_clf.shape],
})


In [ ]:
modelos_clasificacion = {
    "regresion_logistica": LogisticRegression(max_iter=3000),
    "knn": KNeighborsClassifier(n_neighbors=7),
    "arbol_decision": DecisionTreeClassifier(max_depth=6, random_state=42),
    "naive_bayes": GaussianNB(),
}

f1_confirmed = make_scorer(f1_score, pos_label="CONFIRMED", zero_division=0)
resultados_clf = []
modelos_entrenados_clf = {}

for nombre, modelo in modelos_clasificacion.items():
    pipe = Pipeline(
        steps=[
            ("preprocesador", make_numeric_preprocessor(features_clasificacion)),
            ("modelo", modelo),
        ]
    )
    pipe.fit(X_train_clf, y_train_clf)
    pred = pipe.predict(X_test_clf)
    cv_f1 = cross_val_score(pipe, X_train_clf, y_train_clf, cv=5, scoring=f1_confirmed)

    resultados_clf.append(
        {
            "modelo": nombre,
            "Accuracy": accuracy_score(y_test_clf, pred),
            "Precision": precision_score(y_test_clf, pred, pos_label="CONFIRMED", zero_division=0),
            "Recall": recall_score(y_test_clf, pred, pos_label="CONFIRMED", zero_division=0),
            "F1": f1_score(y_test_clf, pred, pos_label="CONFIRMED", zero_division=0),
            "CV_F1_promedio": cv_f1.mean(),
            "CV_F1_desv": cv_f1.std(),
        }
    )
    modelos_entrenados_clf[nombre] = pipe

resultados_clf = pd.DataFrame(resultados_clf).sort_values("F1", ascending=False)
resultados_clf


In [ ]:
mejor_clf = resultados_clf.iloc[0]["modelo"]
mejor_modelo_clf = modelos_entrenados_clf[mejor_clf]
pred_mejor = mejor_modelo_clf.predict(X_test_clf)

print("Mejor modelo:", mejor_clf)
pd.DataFrame(
    {
        "metrica": ["Accuracy", "Precision", "Recall", "F1"],
        "formula_diapositiva": [
            "(TP + TN) / total",
            "TP / (TP + FP)",
            "TP / (TP + FN)",
            "2 * P * R / (P + R)",
        ],
        "valor": [
            accuracy_score(y_test_clf, pred_mejor),
            precision_score(y_test_clf, pred_mejor, pos_label="CONFIRMED", zero_division=0),
            recall_score(y_test_clf, pred_mejor, pos_label="CONFIRMED", zero_division=0),
            f1_score(y_test_clf, pred_mejor, pos_label="CONFIRMED", zero_division=0),
        ],
    }
)


In [ ]:
labels = ["CONFIRMED", "NO_CONFIRMED"]
cm = confusion_matrix(y_test_clf, pred_mejor, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Real_{x}" for x in labels], columns=[f"Predicho_{x}" for x in labels])

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Matriz de confusion - {mejor_clf}",
    template=PLOTLY_TEMPLATE,
)
fig.show()
cm_df


## 5. Regresion: `log1p(koi_prad)`

`koi_prad` tiene outliers muy grandes. Por eso se modela `log1p(koi_prad)`. La evaluacion usa MSE y R2, que son las metricas de regresion mencionadas en la clase.


In [ ]:
regresion_df = kepler[features_regresion + [target_regresion]].dropna(subset=[target_regresion]).copy()
X_reg = regresion_df[features_regresion]
y_reg = np.log1p(regresion_df[target_regresion])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42,
)

pd.DataFrame({
    "particion": ["X_train_reg", "X_test_reg", "y_train_reg", "y_test_reg"],
    "shape": [X_train_reg.shape, X_test_reg.shape, y_train_reg.shape, y_test_reg.shape],
})


In [ ]:
modelos_regresion = {
    "regresion_lineal": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.001, max_iter=10000),
}

resultados_reg = []
modelos_entrenados_reg = {}

for nombre, modelo in modelos_regresion.items():
    pipe = Pipeline(
        steps=[
            ("preprocesador", make_numeric_preprocessor(features_regresion)),
            ("modelo", modelo),
        ]
    )
    pipe.fit(X_train_reg, y_train_reg)
    pred_log = pipe.predict(X_test_reg)
    pred_original = np.expm1(pred_log)
    y_test_original = np.expm1(y_test_reg)
    cv_r2 = cross_val_score(pipe, X_train_reg, y_train_reg, cv=5, scoring="r2")
    resultados_reg.append(
        {
            "modelo": nombre,
            "MSE_log": mean_squared_error(y_test_reg, pred_log),
            "R2_log": r2_score(y_test_reg, pred_log),
            "MSE_radio_tierra": mean_squared_error(y_test_original, pred_original),
            "CV_R2_promedio": cv_r2.mean(),
            "CV_R2_desv": cv_r2.std(),
        }
    )
    modelos_entrenados_reg[nombre] = pipe

resultados_reg = pd.DataFrame(resultados_reg).sort_values("R2_log", ascending=False)
resultados_reg


In [ ]:
mejor_reg = resultados_reg.iloc[0]["modelo"]
mejor_modelo_reg = modelos_entrenados_reg[mejor_reg]
pred_log = mejor_modelo_reg.predict(X_test_reg)
comparacion_reg = pd.DataFrame({
    "y_real_radio_tierra": np.expm1(y_test_reg),
    "y_pred_radio_tierra": np.expm1(pred_log),
    "y_real_log": y_test_reg,
    "y_pred_log": pred_log,
})
comparacion_reg.head(15)


## 6. Cierre de capa de modelado

- La clasificacion usa `koi_disposition`, pero se convierte a una tarea binaria `CONFIRMED` vs `NO_CONFIRMED` para aplicar directamente las formulas de la diapositiva.
- Se comparan algoritmos vistos en clase: regresion logistica, k-NN, arbol de decision y Naive Bayes.
- Las metricas usadas son las de la diapositiva: Accuracy, Precision, Recall, F1 y matriz de confusion.
- La regresion usa una transformacion logaritmica para reducir el efecto de outliers.
- El preprocesamiento se ajusta dentro del pipeline con train, evitando data leakage.
